# 00 - Environment Check

This notebook inspects the core Atlas integrations exposed to JupyterHub and runs bounded connectivity probes for the services listed below. Optional services may be unavailable when their SOURCE is disabled.

## 1. Coverage

- HTTP probes: LiteLLM, Weaviate, ComfyUI, n8n, SearXNG, and Backend API
- Native-client probes: PostgreSQL, Redis, and Neo4j
- Configuration-only inspection: Supabase API gateway

In [ ]:
import os
import httpx
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("Atlas - Environment Check\n")
print("=" * 60)

## 2. Environment Variables

In [ ]:
service_variables = {
    "LiteLLM (LLM gateway)": ("LITELLM_BASE_URL", False),
    "Weaviate (Vector DB)": ("WEAVIATE_URL", False),
    "Neo4j (Graph DB)": ("NEO4J_URI", False),
    "PostgreSQL (Database)": ("DATABASE_URL", True),
    "Redis (Cache)": ("REDIS_URL", True),
    "ComfyUI (Images)": ("COMFYUI_BASE_URL", False),
    "Supabase (API Gateway)": ("SUPABASE_URL", False),
    "n8n (Workflows)": ("N8N_BASE_URL", False),
    "SearxNG (Search)": ("SEARXNG_URL", False),
    "Backend API": ("BACKEND_API_URL", False),
}

print("\nEnvironment Variables:")
print("-" * 60)
for name, (variable, sensitive) in service_variables.items():
    value = os.getenv(variable)
    display = "configured (value hidden)" if sensitive and value else value or "NOT SET"
    status = "SET " if value else "SKIP"
    print(f"{status} {name:30s} {display}")

## 3. HTTP Service Connectivity

In [ ]:
async def check_http_service(url, endpoint="/"):
    """Check if an HTTP service is reachable AND answers non-error.

    4xx/5xx counts as failure because a gateway that answers 404 on its
    health path is not a working service."""
    try:
        full_url = f"{url.rstrip('/')}{endpoint}"
        async with httpx.AsyncClient(timeout=5.0) as client:
            response = await client.get(full_url)
            return response.status_code < 400, response.status_code
    except Exception as e:
        return False, str(e)

print("\nHTTP Service Connectivity:")
print("-" * 60)

http_checks = [
    ("LiteLLM", "LITELLM_BASE_URL", "/health/liveliness"),
    ("Weaviate", "WEAVIATE_URL", "/v1/.well-known/ready"),
    ("ComfyUI", "COMFYUI_BASE_URL", "/system_stats"),
    ("n8n", "N8N_BASE_URL", "/healthz"),
    ("SearxNG", "SEARXNG_URL", "/healthz"),
    ("Backend API", "BACKEND_API_URL", "/health"),
]

for name, variable, endpoint in http_checks:
    url = os.getenv(variable)
    if not url:
        print(f"SKIP {name}: {variable} is not set")
        continue
    success, result = await check_http_service(url, endpoint)
    status = "PASS" if success else "FAIL"
    print(f"{status} {name}: {result}")

## 4. Database Connectivity

In [ ]:
print("\nDatabase Connectivity:")
print("-" * 60)

# Check PostgreSQL
try:
    from sqlalchemy import create_engine, text
    db_url = os.getenv("DATABASE_URL")
    if db_url:
        engine = create_engine(db_url, connect_args={"connect_timeout": 5})
        with engine.connect() as conn:
            result = conn.execute(text("SELECT version()"))
            version = result.fetchone()[0]
            print(f"PASS PostgreSQL: Connected ({version[:30]}...)")
    else:
        print("SKIP PostgreSQL: DATABASE_URL not set")
except Exception as e:
    print(f"FAIL PostgreSQL: {e}")

# Check Redis
try:
    import redis
    redis_url = os.getenv("REDIS_URL")
    if redis_url:
        r = redis.from_url(redis_url, socket_connect_timeout=5, socket_timeout=5)
        r.ping()
        print("PASS Redis: Connected")
    else:
        print("SKIP Redis: REDIS_URL not set")
except Exception as e:
    print(f"FAIL Redis: {e}")

# Check Neo4j
try:
    from neo4j import GraphDatabase
    neo4j_uri = os.getenv("NEO4J_URI")
    neo4j_user = os.getenv("NEO4J_USER")
    neo4j_pass = os.getenv("NEO4J_PASSWORD")
    
    if neo4j_uri and neo4j_user and neo4j_pass:
        with GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_pass), connection_timeout=5) as driver:
            driver.verify_connectivity()
            print("PASS Neo4j: Connected")
    else:
        print("SKIP Neo4j: Connection details not set")
except Exception as e:
    print(f"FAIL Neo4j: {e}")

## 5. Summary

Review every `FAIL` result. A `SKIP` result means the corresponding endpoint or credentials were not injected. Verify:
1. Service is enabled in `.env` (not set to `disabled`)
2. Service is running: `docker compose ps`
3. Environment variables are correctly set

## 6. Next Steps

- `01_litellm_basics.ipynb` - LLM inference via the LiteLLM gateway (Ollama upstream)
- `02_langchain_rag.ipynb` - Build a RAG pipeline with Weaviate
- `03_neo4j_graphs.ipynb` - Work with knowledge graphs
- `04_supabase_data.ipynb` - Database and storage operations
- `05_comfyui_images.ipynb` - Generate images with AI
- `06_n8n_workflows.ipynb` - Automate workflows
- `07_ray_cluster.ipynb` - Distributed compute on the Ray cluster
- `08_scala_basics.ipynb` - Scala basics on the scala3 kernel
- `09_spark_connect.ipynb` - Spark Connect from Python
- `10_spark_scala.ipynb` - Spark Connect from the Scala 2.13 kernel
- `11_financial_research_kit.ipynb` - Read-only OpenBB/CCXT research and paper portfolios
- `12_iceberg_advanced_sql.ipynb` - Advanced Iceberg SQL smoke through Spark Connect
- `13_chonkie_chunking.ipynb` - Compare Chonkie chunking strategies and the Backend `/api/chunk` endpoint
- `14_ragas_evaluation.ipynb` - Evaluate RAG answers with Ragas metrics and the Backend `/api/rag/evaluate` endpoint